# Belief-Driven Software Engineering Agents

### A SPAR demo on Stateful SWE-bench, with a path to OpenTAMP integration

This notebook demonstrates a coding agent whose loop is structured around explicit *belief over user preferences*. We show that the agent can detect when two soft constraints conflict on a given task, ask the user to clarify, and produce a patch that satisfies both the hard test suite and the user's preferred resolution. The framework is designed to drop into [OpenTAMP](https://github.com/Algorithmic-Alignment-Lab/openTAMP)'s hierarchical task-and-motion-planning loop as the *SWE-TAMP* domain.

**Author:** Anusha Mujumdar and Teanna Sims, SPAR alignment research. Mentor: Phillip Christoffersen and Dylan Hadfield-Menell, MIT CSAIL.

**Task instance:** [`matplotlib__matplotlib-22865`](https://github.com/matplotlib/matplotlib/pull/22865), a colorbar rendering bug from the Stateful SWE-bench corpus (a re-framing of SWE-bench Verified instances with user profiles and prior conversation histories; see [ToM-SWE](https://arxiv.org/abs/2510.21903)).

**What is real vs. mocked**

| Aspect | Status |
| --- | --- |
| Task description | Real, paraphrased from the upstream issue and SWE-bench problem statement |
| Failing test | Real, `test_colorbar_extend_drawedges` parametrized over `extend` modes |
| Target method (`_add_solids`) | Real, the actual buggy slice expression from matplotlib 3.5.1 |
| Gold patch shape | Real, matches PR #22865 |
| Surrounding `Colorbar` class | Mocked, we monkey-patch live matplotlib rather than vendoring the full file |
| Developer profile and prior sessions | Synthetic, hand-authored in the style of ToM-SWE's profile schema |
| OpenTAMP integration | Scaffolded, dataclasses and action functions match the planned SWE-TAMP JSON action schema; the planning loop itself is not yet wired up |

**References**

- [SWE-bench Verified](https://www.swebench.com), the underlying instance corpus
- [ToM-SWE](https://arxiv.org/abs/2510.21903), Stateful and Ambiguous SWE-bench
- [OpenTAMP](https://github.com/Algorithmic-Alignment-Lab/openTAMP), the hierarchical planner this work integrates with
- [`opentamp/minimal_tamp/backtrack_plan_recursive.py`](minimal_tamp/backtrack_plan_recursive.py), the planning loop we target
- [`opentamp/new_specs/nav_domain_belief/`](new_specs/nav_domain_belief/), the navigation-domain template


## 1. Setup

We use Claude Sonnet 4.6 via the OpenRouter API. Set `OPENROUTER_API_KEY` in your environment (or, in Colab, store it as a userdata secret).


In [ ]:
# Install dependencies. Safe to skip if already installed.
%pip install -q openai matplotlib numpy pytest


In [ ]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
import tempfile
import textwrap
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any

import matplotlib
import matplotlib.colorbar as mcb
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 110})

# Canonical post-fix _add_solids from matplotlib PR #22865. We use this as
# the deterministic "known good" implementation so the notebook is robust
# to re-runs and to Colab's habit of preserving runtime state across
# notebook reloads. The bug installation, the agent's patches, and the
# safety restore all reference this string explicitly.
GOLD_ADD_SOLIDS_SRC = """
def _add_solids(self, X, Y, C):
    if self.solids is not None:
        self.solids.remove()
    for solid in self.solids_patches:
        solid.remove()
    self.solids_patches = []
    mappable = getattr(self, 'mappable', None)
    if (isinstance(mappable, matplotlib.contour.ContourSet)
            and any(hatch is not None for hatch in mappable.hatches)):
        self._add_solids_patches(X, Y, C, mappable)
    else:
        self.solids = self.ax.pcolormesh(
            X, Y, C, cmap=self.cmap, norm=self.norm, alpha=self.alpha,
            edgecolors='none', shading='flat')
        if not self.drawedges:
            if len(self._y) >= self.n_rasterize:
                self.solids.set_rasterized(True)
    if self.drawedges:
        start_idx = 0 if self._extend_lower() else 1
        end_idx = len(X) if self._extend_upper() else -1
        self.dividers.set_segments(np.dstack([X, Y])[start_idx:end_idx])
    else:
        self.dividers.set_segments([])
"""

def install_gold() -> None:
    """Install the canonical post-fix implementation on Colorbar."""
    ns = {"matplotlib": matplotlib, "np": np}
    exec(GOLD_ADD_SOLIDS_SRC, ns)
    mcb.Colorbar._add_solids = ns["_add_solids"]

# Install the gold patch at import time so any subsequent rendering uses a
# known-good implementation, regardless of what state the runtime was in.
install_gold()


In [ ]:
# OpenRouter API client. The SDK is OpenAI-compatible.
from openai import OpenAI

def _get_api_key() -> str:
    if "OPENROUTER_API_KEY" in os.environ:
        return os.environ["OPENROUTER_API_KEY"]
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get("OPENROUTER_API_KEY")
    except Exception as exc:
        raise RuntimeError(
            "OPENROUTER_API_KEY not found. Set it as an env var or a Colab "
            "userdata secret."
        ) from exc

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=_get_api_key(),
)

MODEL = "anthropic/claude-sonnet-4.6"

def llm(prompt: str, system: str | None = None, json_mode: bool = False,
        max_tokens: int = 2048, temperature: float = 0.0) -> str:
    """One-shot LLM call. Returns the assistant's text response."""
    messages: list[dict[str, Any]] = []
    if system is not None:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    kwargs: dict[str, Any] = {
        "model": MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content or ""


def llm_json(prompt: str, system: str | None = None, max_tokens: int = 2048) -> dict:
    """LLM call constrained to return a JSON object."""
    raw = llm(prompt, system=system, json_mode=True, max_tokens=max_tokens)
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    return json.loads(match.group(0) if match else raw)


## 2. The task: `matplotlib__matplotlib-22865`

**Bug.** When a colorbar is constructed with `drawedges=True` and `extend` set to anything other than `'neither'`, the divider lines between segments are not drawn at the *extended* (triangular) ends. The bug lives in `Colorbar._add_solids`, which slices the divider segments as `np.dstack([X, Y])[1:-1]`, always trimming the first and last segments regardless of whether those ends are extended or not.

**Failing test.** A parametrized pytest function `test_colorbar_extend_drawedges` exercises four cases, `extend='both' | 'min' | 'max' | 'neither'`, and asserts the number of divider segments matches what the extend mode requires.

**Why it matters.** Multiple equally-correct patches exist that look very different. The minimal terse fix is three lines; an "explicit" version with named intermediates is twelve. Both pass the test. Which one is "right" depends on file-style conventions and the user's verbosity preference, exactly the kind of task where soft-constraint reasoning matters.


In [ ]:
# Render the canonical (post-fix) colorbar as a reference. We install the
# gold patch first so the cell is robust to re-runs after the bug has been
# monkey-patched in elsewhere.
install_gold()

fig, ax = plt.subplots(figsize=(1.5, 4))
boundaries = np.array([0, 1, 2, 3, 4, 5])
cmap = plt.get_cmap("viridis", len(boundaries) - 1)
norm = matplotlib.colors.BoundaryNorm(boundaries, cmap.N)
cb = mcb.ColorbarBase(
    ax, cmap=cmap, norm=norm,
    boundaries=[-1] + list(boundaries) + [6],
    extend="both", drawedges=True, spacing="uniform",
)
ax.set_title("reference render\n(fixed matplotlib)", fontsize=9)
plt.tight_layout()
plt.show()


The black divider lines should appear between every adjacent color band, *including* the two triangular extends. In a pre-fix matplotlib those lines are missing at the triangles. The fix lives in `Colorbar._add_solids`, near this line:

```python
self.dividers.set_segments(np.dstack([X, Y])[1:-1] if self.drawedges else [])
```

The `[1:-1]` slice is wrong when either end is extended: it always drops the first and last segment, even when those *are* the extends we wanted to draw.


## 3. Surgical mock and test harness

Rather than vendor the full ~2000 LOC `lib/matplotlib/colorbar.py`, we monkey-patch the live matplotlib install. The reintroduced bug is the *exact* slice expression from pre-fix matplotlib 3.5.1. The agent's task is to replace `_add_solids` with a corrected version; we evaluate via a real pytest subprocess against the same parametrized test the SWE-bench instance ships with.

This means (a) the bug renders for real, (b) the test failure is genuine, and (c) the agent's patch executes against actual matplotlib internals.


In [ ]:
# The "fixed" implementation we restore to is the deterministic gold patch
# from PR #22865, installed earlier. restore_fix() reinstalls it.

BUGGY_ADD_SOLIDS_SRC = """
def _add_solids(self, X, Y, C):
    \"\"\"Draw the colors; optionally add separators.\"\"\"
    if self.solids is not None:
        self.solids.remove()
    for solid in self.solids_patches:
        solid.remove()
    self.solids_patches = []
    mappable = getattr(self, 'mappable', None)
    if (isinstance(mappable, matplotlib.contour.ContourSet)
            and any(hatch is not None for hatch in mappable.hatches)):
        self._add_solids_patches(X, Y, C, mappable)
    else:
        self.solids = self.ax.pcolormesh(
            X, Y, C, cmap=self.cmap, norm=self.norm, alpha=self.alpha,
            edgecolors='none', shading='flat')
        if not self.drawedges:
            if len(self._y) >= self.n_rasterize:
                self.solids.set_rasterized(True)
    # BUG: the [1:-1] slice always drops the first and last segments, even
    # when the corresponding end is extended. Fixed in matplotlib PR #22865.
    self.dividers.set_segments(
        np.dstack([X, Y])[1:-1] if self.drawedges else [])
"""

def install_bug() -> None:
    """Patch in the buggy version of _add_solids."""
    ns = {"matplotlib": matplotlib, "np": np}
    exec(BUGGY_ADD_SOLIDS_SRC, ns)
    mcb.Colorbar._add_solids = ns["_add_solids"]

def restore_fix() -> None:
    """Reinstall the canonical post-fix implementation from PR #22865."""
    install_gold()

def apply_patch(method_src: str) -> None:
    """Install an agent-produced _add_solids method on the live class."""
    ns = {"matplotlib": matplotlib, "np": np}
    exec(method_src, ns)
    mcb.Colorbar._add_solids = ns["_add_solids"]


In [ ]:
# The real failing test from PR #22865, embedded as source for the harness.
FAILING_TEST_SRC = """
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colorbar as mcb
import numpy as np
import pytest


@pytest.mark.parametrize('extend,expected_segments', [
    ('both', 7),
    ('min', 6),
    ('max', 6),
    ('neither', 5),
])
def test_colorbar_extend_drawedges(extend, expected_segments):
    fig, ax = plt.subplots(figsize=(1, 3))
    boundaries = np.array([0, 1, 2, 3, 4, 5])
    cmap = plt.get_cmap('viridis', len(boundaries) - 1)
    norm = matplotlib.colors.BoundaryNorm(boundaries, cmap.N)
    cb = mcb.ColorbarBase(
        ax, cmap=cmap, norm=norm,
        boundaries=[-1] + list(boundaries) + [6],
        extend=extend, drawedges=True, spacing='uniform',
    )
    plt.close(fig)
    assert len(cb.dividers.get_segments()) == expected_segments
"""

def run_test_against(method_src: str) -> dict:
    """Write the candidate _add_solids and the failing test to a temp dir,
    run pytest as a subprocess, return structured results."""
    with tempfile.TemporaryDirectory() as tmpdir:
        conftest = textwrap.dedent("""
            import matplotlib
            import matplotlib.colorbar
            import numpy as np
            _ns = {"matplotlib": matplotlib, "np": np}
            with open("patch.py") as f:
                exec(f.read(), _ns)
            matplotlib.colorbar.Colorbar._add_solids = _ns["_add_solids"]
        """).strip()
        Path(tmpdir, "patch.py").write_text(method_src)
        Path(tmpdir, "conftest.py").write_text(conftest)
        Path(tmpdir, "test_swe.py").write_text(FAILING_TEST_SRC)
        proc = subprocess.run(
            [sys.executable, "-m", "pytest", "-v", "--no-header",
             "--tb=short", "test_swe.py"],
            cwd=tmpdir, capture_output=True, text=True, timeout=60,
        )
    per_test = {}
    for line in proc.stdout.splitlines():
        m = re.match(r"test_swe\.py::(\S+)\s+(PASSED|FAILED)", line)
        if m:
            per_test[m.group(1)] = (m.group(2) == "PASSED")
    return {
        "all_passed": proc.returncode == 0,
        "exit_code": proc.returncode,
        "stdout": proc.stdout,
        "stderr": proc.stderr,
        "per_test": per_test,
        "n_passed": sum(per_test.values()),
        "n_total": len(per_test),
    }


In [ ]:
# Sanity check: confirm the buggy implementation fails the test.
res = run_test_against(BUGGY_ADD_SOLIDS_SRC)
print(f"Buggy implementation: {res['n_passed']}/{res['n_total']} tests passed.")
for name, ok in res["per_test"].items():
    mark = "PASS" if ok else "FAIL"
    print(f"  [{mark}] {name}")


## 4. OpenTAMP-compatible scaffolding

The dataclasses and action functions below mirror the planned SWE-TAMP domain. When wired into [`backtrack_plan_recursive.py`](minimal_tamp/backtrack_plan_recursive.py), the dataclasses become JSON action-schema objects and the action functions become OpenTAMP action samplers. Control flow mirrors `nav_2d_belief_plan`'s action skeleton.

**Predicate and action mapping**

| `nav_2d_belief_plan` | SWE-TAMP equivalent | Meaning |
| --- | --- | --- |
| `point_at_obs(Robot, Obstacle)` | `read_session_history(Agent, Constraint)` | Direct attention at a constraint |
| `observe_obs(Robot, Obstacle)` | `clarify_constraint(Agent, Constraint)` | Resolve uncertainty over a soft constraint |
| `observe_targ(Robot, Target)` | `clarify_intent(Agent, CodingTask)` | Resolve uncertainty over task intent |
| `confirm_targ(Robot, Target)` | `confirm_spec(Agent, CodingTask)` | Lock the spec before executing |
| `move_avoid(Robot, Target, Soft, Obs...)` | `execute_code_action(Agent, Task, Milestone, Constraints...)` | Produce a patch satisfying all certain constraints |
| `CertainObs(o)` | `CertainConstraint(c)` | Soft constraint resolved |
| `CertainTarget(t)` | `CertainIntent(t)` | Task intent resolved |
| `ConfirmedTarget(t)` | `ConfirmedSpec(t)` | Spec locked |
| `RobotAtTarget(r, t)` | `TaskComplete(a, t)` | Goal predicate |

The critical *forall* precondition is preserved: `execute_code_action` cannot fire until every `Constraint` has been resolved to `CertainConstraint`. This forces information-gathering before action, analogous to how `move_avoid` cannot fire until every obstacle is observed.


In [ ]:
@dataclass(frozen=True)
class Constraint:
    """A soft user constraint (preference). Maps to Obstacle in nav_2d."""
    name: str
    kind: str  # "stylistic" | "architectural" | "testing" | "org"
    statement: str

    def __str__(self) -> str:
        return f'[{self.kind}] "{self.statement}"'


@dataclass
class CodingTask:
    """The task the agent is solving. Maps to Target in nav_2d."""
    instance_id: str
    problem_statement: str
    target_method: str
    failing_test: str


@dataclass
class ParticleBelief:
    """Discrete categorical belief over which constraint is dominant in the
    user's intent for this task. Particles are indices into the constraint
    list; weights sum to one. This is the analogue of nav_2d's spatial
    belief over obstacle positions."""
    constraints: list[Constraint]
    weights: np.ndarray
    n_particles: int = 200

    @classmethod
    def uniform(cls, constraints: list[Constraint], n_particles: int = 200) -> "ParticleBelief":
        n = len(constraints)
        return cls(constraints, np.ones(n) / n, n_particles)

    def particles(self) -> np.ndarray:
        return np.random.choice(
            len(self.constraints), size=self.n_particles, p=self.weights,
        )

    def update(self, evidence: dict[str, float]) -> "ParticleBelief":
        """Multiplicative update. `evidence` maps constraint name to a
        likelihood ratio (>1 means evidence in favor)."""
        new_w = self.weights.copy()
        for i, c in enumerate(self.constraints):
            new_w[i] *= evidence.get(c.name, 1.0)
        new_w = new_w / new_w.sum()
        return ParticleBelief(self.constraints, new_w, self.n_particles)


@dataclass
class AgentState:
    """Symbolic state the planner reasons over."""
    certain_constraint: set[str] = field(default_factory=set)
    certain_intent: bool = False
    confirmed_spec: bool = False
    completed_action: bool = False
    tests_passing: bool = False

    def can_execute(self, constraints: list[Constraint]) -> bool:
        """The forall precondition on execute_code_action."""
        return (self.confirmed_spec and
                all(c.name in self.certain_constraint for c in constraints))


In [ ]:
# Action stubs. Each action mutates AgentState and returns the new state.
# Filled in in later sections.

def read_session_history(state, history, constraints):
    """Action: load prior user context into a non-uniform prior over constraints."""
    raise NotImplementedError("Implemented in section 5.")


def clarify_constraint(state, belief, constraints, task):
    """Action: detect and resolve conflict over soft constraints via an LLM judge."""
    raise NotImplementedError("Implemented in section 7.")


def confirm_spec(state, task):
    """Action: lock the spec once intent is certain."""
    if not state.certain_intent:
        raise RuntimeError("confirm_spec requires certain_intent")
    new = AgentState(**asdict(state))
    new.confirmed_spec = True
    return new


def execute_code_action(state, task, constraints, belief, resolution):
    """Action: produce a patch. Forall precondition: all constraints certain."""
    if not state.can_execute(constraints):
        raise RuntimeError("execute_code_action precondition violated")
    raise NotImplementedError("Implemented in section 7.")


## 5. Developer profile and prior session history

Stateful SWE-bench pairs each task with a synthesized developer profile and prior conversation sessions. The agent reads this state first (the `read_session_history` action). We hand-author a profile in the style of ToM-SWE's schema.

### Profile

**Alex Chen**, Senior platform engineer at a mid-size fintech. Six years of Python, three on a large internal data-visualization library. Frequently reviews PRs and pushes back hard on patches that introduce stylistic inconsistency with the surrounding file. Prefers terse, idiomatic NumPy over named-intermediate expansions, but defers to local readability when the surrounding code is already verbose.

**Stated coding preferences**

- Match the existing style of the file you are editing.
- Prefer five lines of dense NumPy over fifteen lines of variable assignments, unless the dense version genuinely obscures the logic.
- Do not add helper methods unless the same logic recurs in two or more places.
- Add a regression test for every bugfix.

**Stated interactional preferences**

- One focused question if something is genuinely ambiguous. Not three.
- If there is an obvious right answer, just do it and report what you did.


### Prior session transcripts (synthesized)

These three excerpts are what `read_session_history` consumes to initialise the belief. They are styled as chat transcripts following ToM-SWE's session format.


In [ ]:
@dataclass
class PriorTurn:
    session_id: str
    role: str  # "user" | "agent"
    content: str


@dataclass
class SessionHistory:
    profile_name: str
    profile_summary: str
    turns: list[PriorTurn]

    def render_markdown(self) -> str:
        out = [f"### Developer: {self.profile_name}\n",
               f"_{self.profile_summary}_\n"]
        last = None
        for t in self.turns:
            if t.session_id != last:
                out.append(f"\n**Session {t.session_id}**\n")
                last = t.session_id
            tag = "**Alex:**" if t.role == "user" else "**Agent:**"
            out.append(f"{tag} {t.content}\n")
        return "\n".join(out)


ALEX_HISTORY = SessionHistory(
    profile_name="Alex Chen",
    profile_summary=(
        "Senior platform engineer, fintech. Values file-style consistency. "
        "Defers to local readability when surrounding code is verbose."
    ),
    turns=[
        PriorTurn("S-114", "user",
            "On this PR you replaced the three-line numpy slice with a six-line "
            "version with named variables. I appreciate the readability impulse "
            "but the rest of this module is one-liners. Keep it consistent with "
            "the file, terse here is fine."),
        PriorTurn("S-114", "agent",
            "Understood. Reverted to the original slice form and pushed."),

        PriorTurn("S-138", "user",
            "I am fine with you expanding things into named intermediates when "
            "the function is already long and conditional. But for short "
            "slice-style edits in numpy-heavy files, match what is around you."),
        PriorTurn("S-138", "agent",
            "Acknowledged. Heuristic: match the file's verbosity unless the "
            "surrounding logic is already verbose."),

        PriorTurn("S-147", "user",
            "Last thing: when you fix a bug, add a parametrized test covering "
            "the cases the fix handles. Even if the bug feels obvious. I want "
            "regressions caught."),
        PriorTurn("S-147", "agent",
            "Will do. Parametrized regression test for every bugfix."),
    ],
)

display(Markdown(ALEX_HISTORY.render_markdown()))


In [ ]:
# Implement read_session_history. The action turns prior conversational
# evidence into a non-uniform prior over which constraint is dominant.

def read_session_history(state: AgentState, history: SessionHistory,
                         constraints: list[Constraint]) -> tuple[AgentState, ParticleBelief]:
    """LLM-score how strongly the prior session evidence endorses each
    candidate constraint. Returns updated state plus belief."""
    sys_prompt = (
        "You score how strongly a developer endorses each candidate coding "
        "constraint, based on a transcript of their prior sessions. Output a "
        "JSON object with key 'scores' mapping constraint name to a real "
        "number in [0, 1]. Higher means the prior turns more strongly endorse "
        "that constraint. Be calibrated; do not flatten."
    )
    user_prompt = (
        f"Developer profile: {history.profile_summary}\n\n"
        f"Prior session transcript:\n{history.render_markdown()}\n\n"
        "Candidate constraints:\n" +
        "\n".join(f"- {c.name}: {c.statement}" for c in constraints) +
        "\n\nReturn JSON: "
        '{"scores": {"<constraint_name>": <score in [0,1]>, ...}}'
    )
    result = llm_json(user_prompt, system=sys_prompt)
    scores = result["scores"]
    weights = np.array([float(scores.get(c.name, 0.5)) for c in constraints])
    weights = weights / weights.sum()
    new_state = AgentState(**asdict(state))
    return new_state, ParticleBelief(constraints, weights)


In [ ]:
# Visualization helper for belief over constraints.
def plot_belief(belief: ParticleBelief, title: str, ax=None):
    """Horizontal bar chart of weights over constraint names."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 0.5 + 0.35 * len(belief.constraints)))
    names = [c.name for c in belief.constraints]
    kinds = [c.kind for c in belief.constraints]
    kind_color = {"stylistic": "#4C72B0", "architectural": "#DD8452",
                  "testing": "#55A467", "org": "#8172B3"}
    colors = [kind_color.get(k, "#888") for k in kinds]
    y = np.arange(len(names))
    ax.barh(y, belief.weights, color=colors, edgecolor="black", linewidth=0.5)
    ax.set_yticks(y)
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel("posterior weight")
    ax.set_xlim(0, max(belief.weights.max() * 1.15, 0.05))
    ax.set_title(title, fontsize=10)
    ax.invert_yaxis()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    return ax


## 6. Baseline: flat agent

The flat agent has no notion of belief, no clarification action, no symbolic precondition stack. It sees the task, the failing test, and a flat list of preferences, and iterates `propose -> test -> feedback` until either all tests pass or it gives up.

This is the standard coding-agent loop. We run it on the colorbar task to establish the baseline before introducing belief-driven planning.


In [ ]:
# Define the task. The problem_statement matches the paraphrase used by the
# SWE-bench instance; the failing_test is the exact body from PR #22865.
TASK = CodingTask(
    instance_id="matplotlib__matplotlib-22865",
    problem_statement=(
        "When a matplotlib Colorbar is constructed with drawedges=True and "
        "extend set to 'both', 'min', or 'max', the divider lines that "
        "separate adjacent color bands are not drawn at the extended "
        "(triangular) ends. The slice expression "
        "np.dstack([X, Y])[1:-1] in Colorbar._add_solids unconditionally "
        "drops the first and last segments, even when those ends are "
        "extended and should have their dividers drawn. Fix _add_solids so "
        "that divider segments are kept for any extended end."
    ),
    target_method="matplotlib.colorbar.Colorbar._add_solids",
    failing_test=FAILING_TEST_SRC.strip(),
)


In [ ]:
# The full preference list. Pair A is buried among distractors for the
# emergent-detection run later. For the scripted run we use the first two.
PREFS_ALL = [
    Constraint(
        name="match_file_style",
        kind="org",
        statement="Match the existing style of the file you are editing.",
    ),
    Constraint(
        name="prefer_explicit_names",
        kind="stylistic",
        statement="Prefer explicit, named intermediate variables over terse one-liners.",
    ),
    Constraint(
        name="regression_test_every_bugfix",
        kind="testing",
        statement="Add a parametrized regression test for every bugfix.",
    ),
    Constraint(
        name="no_new_helpers_unless_reused",
        kind="architectural",
        statement="Do not add helper methods unless the same logic recurs in two or more places.",
    ),
    Constraint(
        name="add_type_annotations",
        kind="stylistic",
        statement="Add type annotations to functions you touch.",
    ),
    Constraint(
        name="minimize_pr_churn",
        kind="org",
        statement="Minimize PR churn, change only what the bug requires.",
    ),
    Constraint(
        name="prefer_composition_over_inheritance",
        kind="architectural",
        statement="Prefer composition over inheritance when restructuring.",
    ),
    Constraint(
        name="avoid_gpl_dependencies",
        kind="org",
        statement="Avoid introducing GPL-licensed dependencies.",
    ),
]

PAIR_A = [PREFS_ALL[0], PREFS_ALL[1]]  # scripted-conflict subset
PAIR_A


In [ ]:
FLAT_SYSTEM_PROMPT = (
    "You are a Python coding assistant working on a matplotlib bug. "
    "You are given a target method to rewrite, a description of the bug, "
    "a failing test, and a list of user coding preferences. "
    "Return ONLY a Python code block defining a single top-level function "
    "named `_add_solids` with signature `(self, X, Y, C)`. No prose."
)


def flat_agent(task: CodingTask, constraints: list[Constraint],
               max_attempts: int = 4) -> list[dict]:
    """Vanilla propose-test-feedback loop. No belief, no clarification."""
    transcript: list[dict] = []
    feedback = ""
    for attempt in range(1, max_attempts + 1):
        prompt = (
            f"TASK: {task.problem_statement}\n\n"
            f"TARGET METHOD: {task.target_method}\n\n"
            f"FAILING TEST:\n```python\n{task.failing_test}\n```\n\n"
            "USER PREFERENCES (treat as soft constraints):\n" +
            "\n".join(f"- {c}" for c in constraints) +
            (f"\n\nPRIOR FEEDBACK:\n{feedback}" if feedback else "")
        )
        raw = llm(prompt, system=FLAT_SYSTEM_PROMPT, max_tokens=1500)
        m = re.search(r"```python\s*\n(.*?)```", raw, re.DOTALL)
        code_str = m.group(1).strip() if m else raw.strip()
        test_res = run_test_against(code_str)
        transcript.append({
            "attempt": attempt,
            "code": code_str,
            "test": test_res,
        })
        print(f"  [flat attempt {attempt}] "
              f"{test_res['n_passed']}/{test_res['n_total']} passed.")
        if test_res["all_passed"]:
            break
        feedback = "Test output (truncated):\n" + test_res["stdout"][-800:]
    return transcript


In [ ]:
# Run the flat-agent baseline on Pair A. This is the "no belief, no
# clarification" point of comparison.
print("Running flat agent on PAIR A (no clarification, no belief)...")
flat_transcript = flat_agent(TASK, PAIR_A, max_attempts=4)
print(f"\nFlat agent finished after {len(flat_transcript)} attempt(s).")
final_flat = flat_transcript[-1]
print(f"Final result: {final_flat['test']['n_passed']}/"
      f"{final_flat['test']['n_total']} passed.")


In [ ]:
# Visualisation 5: per-test heatmap across attempts for the flat agent.
def plot_test_heatmap(transcript: list[dict], title: str):
    test_names: list[str] = []
    grid: list[list[int]] = []
    for h in transcript:
        per = h["test"]["per_test"]
        if not test_names:
            test_names = list(per.keys())
        grid.append([int(per.get(n, False)) for n in test_names])
    if not grid:
        return
    arr = np.array(grid).T
    fig, ax = plt.subplots(figsize=(0.9 + 0.7 * len(grid), 0.4 * len(test_names) + 1))
    cmap = matplotlib.colors.ListedColormap(["#C44E52", "#55A467"])
    ax.imshow(arr, cmap=cmap, aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(grid)))
    ax.set_xticklabels([f"attempt {i+1}" for i in range(len(grid))])
    ax.set_yticks(range(len(test_names)))
    ax.set_yticklabels(test_names, fontsize=8)
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

plot_test_heatmap(flat_transcript, "Flat agent: per-test pass/fail across attempts")


## 7. Belief-driven agent

We now run the same task through a planner that mirrors `nav_2d_belief_plan`'s action skeleton:

1. `read_session_history`, already done in section 5, gives a non-uniform prior over which preferences are dominant for this developer
2. `clarify_intent`, for this task the intent is given (fix the divider bug), so this is immediate
3. `clarify_constraint`, an LLM judge inspects the task and the constraint list and asks whether any pair would produce visibly different patches; if yes, it generates one focused clarifying question (honouring Alex's interactional preference)
4. The simulated user (oracle) answers, the belief updates, `CertainConstraint` is set for each constraint
5. `confirm_spec`, preconditions met, spec is locked
6. `execute_code_action`, the patch is produced under the resolved constraint ordering, run against the real test, and graded per-preference by a second LLM judge

The forall precondition on `execute_code_action` (`all c: CertainConstraint(c) /\ ConfirmedSpec`) is exactly the analogue of `move_avoid` requiring all obstacles to be `CertainObs` in nav_2d.


In [ ]:
# Step 1 of the belief-driven loop: detect conflict between constraints on this
# specific task. The judge analyses the task and reasons about whether two
# preferences would pull a patch in visibly different directions.

CONFLICT_JUDGE_SYSTEM = (
    "You are a code-review judge. Given a software-engineering task and a list "
    "of soft user preferences, determine whether any pair of preferences would "
    "produce visibly different patches on this specific task. If yes, identify "
    "the most consequential conflicting pair and generate one focused "
    "clarifying question to ask the user. Return JSON only."
)

def detect_conflict(task: CodingTask, belief: ParticleBelief) -> dict:
    """LLM-as-judge over the belief's constraints. Returns a structured
    conflict report with optional clarifying question."""
    cs = belief.constraints
    weighted = sorted(
        zip(cs, belief.weights), key=lambda cw: -cw[1])
    pref_block = "\n".join(
        f"- {c.name} (weight={w:.2f}): {c.statement}" for c, w in weighted)
    user_prompt = (
        f"TASK: {task.problem_statement}\n\n"
        f"TARGET METHOD: {task.target_method}\n\n"
        f"USER PREFERENCES (with current belief weights):\n{pref_block}\n\n"
        "Return JSON: {\n"
        '  "conflict_detected": <bool>,\n'
        '  "pair": [<constraint_name_a>, <constraint_name_b>] or null,\n'
        '  "rationale": <one sentence explaining the conflict>,\n'
        '  "question": <one focused question to ask the user> or null\n'
        "}"
    )
    return llm_json(user_prompt, system=CONFLICT_JUDGE_SYSTEM)


In [ ]:
# Step 2: simulate the user response. ToM-SWE uses an LLM-simulated user; we
# follow that convention. The oracle is conditioned on Alex's profile plus
# prior history, so its answer is consistent with what Alex would say.

ORACLE_SYSTEM = (
    "You are simulating a senior software engineer answering a clarifying "
    "question from a coding agent. Respond in character based on the profile "
    "and prior session transcripts provided. Keep your answer to one or two "
    "sentences. Be decisive, pick one of the alternatives if asked."
)

def user_oracle(question: str, history: SessionHistory, task: CodingTask) -> str:
    user_prompt = (
        f"Your profile and recent sessions:\n{history.render_markdown()}\n\n"
        f"Current task context: {task.problem_statement}\n\n"
        f"The agent asks: {question}\n\n"
        "Respond as Alex would, in one or two sentences."
    )
    return llm(user_prompt, system=ORACLE_SYSTEM, max_tokens=200)


In [ ]:
# Step 3: turn the user response into a belief update. The judge again is the
# best LLM for this: it scores which constraint the answer endorses more.

EVIDENCE_JUDGE_SYSTEM = (
    "You score how strongly a user's answer endorses each of two competing "
    "coding preferences. Output JSON with key 'evidence' mapping each "
    "constraint name to a likelihood ratio: a value > 1 if the answer "
    "favours that constraint, < 1 if it disfavours it, 1 if neutral. "
    "Calibrate: a clear endorsement should be around 5x; a slight lean ~1.5x."
)

def evidence_from_answer(answer: str, pair: list[str],
                         constraints: list[Constraint]) -> dict[str, float]:
    by_name = {c.name: c for c in constraints}
    statements = "\n".join(
        f"- {n}: {by_name[n].statement}" for n in pair)
    user_prompt = (
        f"Competing preferences:\n{statements}\n\n"
        f"User's answer: {answer}\n\n"
        'Return JSON: {"evidence": {"<name>": <likelihood ratio>, ...}}'
    )
    return llm_json(user_prompt, system=EVIDENCE_JUDGE_SYSTEM)["evidence"]


In [ ]:
# Step 4: execute_code_action under resolved constraints. The prompt makes
# the belief-weighted preference ranking explicit so the model satisfies the
# dominant preference first.

EXEC_SYSTEM = (
    "You are a Python coding assistant working on a matplotlib bug. "
    "You are given the CURRENT (buggy) implementation of the method and a "
    "list of user preferences in priority order. Make the MINIMAL "
    "modification that fixes the failing test, while honouring the highest-"
    "priority preferences. Do NOT add references to attributes or methods "
    "that are not already used in the current implementation. Return ONLY a "
    "Python code block defining a single top-level function named "
    "`_add_solids` with signature `(self, X, Y, C)`. No prose."
)

def execute_code_action(task: CodingTask, belief: ParticleBelief,
                        max_attempts: int = 2) -> dict:
    """Generate a patch; run the test; retry once on failure."""
    cs = belief.constraints
    ranked = sorted(zip(cs, belief.weights), key=lambda cw: -cw[1])
    pref_block = "\n".join(
        f"{i+1}. (weight {w:.2f}) {c.statement}"
        for i, (c, w) in enumerate(ranked))
    feedback = ""
    for attempt in range(1, max_attempts + 1):
        prompt = (
            f"TASK: {task.problem_statement}\n\n"
            f"TARGET METHOD: {task.target_method}\n\n"
            f"CURRENT (BUGGY) IMPLEMENTATION:\n"
            f"```python\n{BUGGY_ADD_SOLIDS_SRC.strip()}\n```\n\n"
            f"FAILING TEST:\n```python\n{task.failing_test}\n```\n\n"
            f"USER PREFERENCES (priority order):\n{pref_block}\n\n"
            "Make the minimal modification to the current implementation "
            "that fixes the failing test. Use only attributes and methods "
            "already referenced in the buggy implementation "
            "(`self._extend_lower()`, `self._extend_upper()`, "
            "`self.drawedges`, `self.dividers`, `np.dstack`, etc.). " +
            (f"\nPRIOR FEEDBACK:\n{feedback}\n" if feedback else "")
        )
        raw = llm(prompt, system=EXEC_SYSTEM, max_tokens=1500)
        m = re.search(r"```python\s*\n(.*?)```", raw, re.DOTALL)
        code_str = m.group(1).strip() if m else raw.strip()
        test_res = run_test_against(code_str)
        if test_res["all_passed"]:
            return {"attempt": attempt, "code": code_str, "test": test_res}
        feedback = "Test output (truncated):\n" + test_res["stdout"][-800:]
    return {"attempt": max_attempts, "code": code_str, "test": test_res}


In [ ]:
# Step 5: LLM-as-judge over the final patch against each preference. This is
# the soft-constraint analogue of the hard test suite.

PATCH_JUDGE_SYSTEM = (
    "You are a senior code reviewer scoring a patch against a list of user "
    "preferences. For each preference, return a score in [0, 1] (1 = fully "
    "satisfied) and a one-sentence rationale. Output JSON only."
)

def judge_patch(patch: str, constraints: list[Constraint], task: CodingTask) -> dict:
    pref_block = "\n".join(f"- {c.name}: {c.statement}" for c in constraints)
    user_prompt = (
        f"TASK: {task.problem_statement}\n\n"
        f"PATCH:\n```python\n{patch}\n```\n\n"
        f"PREFERENCES:\n{pref_block}\n\n"
        'Return JSON: {"scores": {"<name>": {"score": <0..1>, '
        '"rationale": "<one sentence>"}, ...}}'
    )
    return llm_json(user_prompt, system=PATCH_JUDGE_SYSTEM)["scores"]


In [ ]:
# Assemble the belief-driven agent. Returns a structured log we can plot.

def belief_agent(task: CodingTask, constraints: list[Constraint],
                 history: SessionHistory, max_clarifications: int = 2,
                 verbose: bool = True) -> dict:
    log: list[dict] = []
    state = AgentState()

    # Action: read_session_history
    state, belief = read_session_history(state, history, constraints)
    log.append({"step": "read_session_history", "belief": belief, "state": asdict(state)})
    if verbose:
        print(f"[read_session_history] prior belief: " +
              ", ".join(f"{c.name}={w:.2f}" for c, w in
                        zip(constraints, belief.weights)))

    # Action: clarify_intent (the task description is unambiguous here)
    state.certain_intent = True
    log.append({"step": "clarify_intent", "belief": belief, "state": asdict(state)})

    # Action: clarify_constraint (repeat until no conflict or budget exhausted)
    clarifications: list[dict] = []
    for round_num in range(1, max_clarifications + 1):
        report = detect_conflict(task, belief)
        if not report.get("conflict_detected"):
            if verbose:
                print(f"[clarify_constraint round {round_num}] no conflict detected.")
            break
        pair = report["pair"]
        question = report["question"]
        if verbose:
            print(f"[clarify_constraint round {round_num}] conflict: "
                  f"{pair[0]} vs {pair[1]}")
            print(f"  question to user: {question}")
        answer = user_oracle(question, history, task)
        if verbose:
            print(f"  user (Alex): {answer}")
        evidence = evidence_from_answer(answer, pair, constraints)
        belief = belief.update(evidence)
        clarifications.append({
            "round": round_num, "pair": pair, "question": question,
            "answer": answer, "evidence": evidence,
        })
        log.append({"step": f"clarify_constraint_{round_num}",
                    "belief": belief, "state": asdict(state),
                    "question": question, "answer": answer})

    # Mark all constraints certain after the clarification budget.
    state.certain_constraint = {c.name for c in constraints}

    # Action: confirm_spec
    state = confirm_spec(state, task)
    log.append({"step": "confirm_spec", "belief": belief, "state": asdict(state)})

    # Action: execute_code_action
    if verbose:
        print("[execute_code_action] generating patch under resolved preferences...")
    exec_res = execute_code_action(task, belief)
    state.completed_action = True
    state.tests_passing = exec_res["test"]["all_passed"]
    log.append({"step": "execute_code_action", "belief": belief,
                "state": asdict(state),
                "patch": exec_res["code"], "test": exec_res["test"]})
    if verbose:
        print(f"  tests: {exec_res['test']['n_passed']}/"
              f"{exec_res['test']['n_total']} passed.")

    # Soft-constraint evaluation.
    pref_scores = judge_patch(exec_res["code"], constraints, task)

    return {
        "log": log, "patch": exec_res["code"], "test": exec_res["test"],
        "pref_scores": pref_scores, "belief_final": belief,
        "clarifications": clarifications,
    }


In [ ]:
# Run the scripted Pair A demo end-to-end.
print("Running belief-driven agent on PAIR A...")
print()
belief_result = belief_agent(TASK, PAIR_A, ALEX_HISTORY,
                              max_clarifications=2, verbose=True)
print()
print(f"Belief agent finished. tests: "
      f"{belief_result['test']['n_passed']}/{belief_result['test']['n_total']}.")


### Visualisation: belief evolution across clarification turns

The belief over which preference is dominant updates after every clarification. The plot below shows the posterior at each step of the action skeleton, from the prior (informed by Alex's session history) through any clarification rounds to the final state used by `execute_code_action`.


In [ ]:
def plot_belief_evolution(log: list[dict], title: str = "Belief evolution"):
    steps = [entry for entry in log if "belief" in entry and
             entry["step"] not in ("clarify_intent", "confirm_spec")]
    n = len(steps)
    fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 0.5 + 0.35 * len(steps[0]["belief"].constraints)),
                             squeeze=False)
    for i, entry in enumerate(steps):
        plot_belief(entry["belief"], entry["step"], ax=axes[0, i])
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()

plot_belief_evolution(belief_result["log"],
                      "Belief over constraints across belief-agent steps (Pair A)")


### Visualisation: action skeleton

The belief-driven agent executes a fixed action skeleton, the same one `backtrack_plan_recursive.py` would receive from FastDownward in the OpenTAMP integration. Each box is one symbolic action; check marks indicate which preconditions were satisfied.


In [ ]:
def plot_action_skeleton(log: list[dict], title: str = "Action skeleton"):
    steps = [
        ("read_session_history", True),
        ("clarify_intent", True),
    ]
    # Add per-round clarification slots
    n_clar = sum(1 for e in log if e["step"].startswith("clarify_constraint"))
    for i in range(1, n_clar + 1):
        steps.append((f"clarify_constraint #{i}", True))
    final_state = log[-1]["state"]
    steps.append(("confirm_spec", final_state["confirmed_spec"]))
    steps.append(("execute_code_action", final_state["completed_action"]))
    steps.append(("TestsPassing", final_state["tests_passing"]))

    fig, ax = plt.subplots(figsize=(1.4 + 1.6 * len(steps), 1.6))
    for i, (name, ok) in enumerate(steps):
        color = "#55A467" if ok else "#C44E52"
        ax.add_patch(plt.Rectangle((i, 0), 0.9, 0.7,
                                    facecolor=color, edgecolor="black",
                                    linewidth=0.8, alpha=0.85))
        mark = "checkmark" if ok else "cross"
        symbol = "✓" if ok else "✗"
        ax.text(i + 0.45, 0.5, symbol, ha="center", va="center",
                fontsize=14, color="white", fontweight="bold")
        ax.text(i + 0.45, 0.85, name, ha="center", va="bottom",
                fontsize=8, wrap=True)
        if i < len(steps) - 1:
            ax.annotate("", xy=(i + 1.0, 0.35), xytext=(i + 0.9, 0.35),
                        arrowprops=dict(arrowstyle="->", color="black", lw=0.8))
    ax.set_xlim(-0.1, len(steps) + 0.05)
    ax.set_ylim(-0.05, 1.2)
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

plot_action_skeleton(belief_result["log"],
                      "Belief-driven agent: action skeleton (Pair A)")


### Visualisation: per-preference satisfaction (LLM judge)

The hard test suite says only whether tests pass. The LLM judge scores how well the final patch honours each soft preference, the analogue of the cost term over soft constraints in continuous-layer trajectory optimization.


In [ ]:
def plot_pref_scores(scores: dict, title: str = "Preference satisfaction"):
    names = list(scores.keys())
    vals = [scores[n]["score"] for n in names]
    fig, ax = plt.subplots(figsize=(7, 0.5 + 0.35 * len(names)))
    y = np.arange(len(names))
    bars = ax.barh(y, vals, color="#4C72B0", edgecolor="black", linewidth=0.5)
    ax.set_yticks(y)
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("judge score")
    ax.set_title(title, fontsize=10)
    ax.invert_yaxis()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    # Annotate with rationale (truncated) above each bar.
    for i, n in enumerate(names):
        rationale = scores[n]["rationale"]
        ax.text(vals[i] + 0.02, i, rationale[:80], va="center", fontsize=7)
    plt.tight_layout()
    plt.show()

plot_pref_scores(belief_result["pref_scores"],
                  "Belief-driven agent: per-preference satisfaction (Pair A)")


## 8. Side-by-side: flat agent vs. belief-driven agent

The flat agent and the belief-driven agent both saw the same task and the same two preferences. The flat agent had no mechanism to detect that the two preferences conflict on this task; the belief agent asked one focused question and proceeded under resolved constraints.


In [ ]:
def show_patches_side_by_side(flat_code: str, belief_code: str):
    from IPython.display import HTML
    def esc(s: str) -> str:
        return (s.replace("&", "&amp;")
                 .replace("<", "&lt;")
                 .replace(">", "&gt;"))
    html = f"""
    <table style='width:100%;border-collapse:collapse;font-family:monospace;font-size:11px;'>
      <tr>
        <th style='border:1px solid #888;padding:6px;background:#eee;text-align:left;width:50%;'>Flat agent patch</th>
        <th style='border:1px solid #888;padding:6px;background:#eee;text-align:left;width:50%;'>Belief-driven agent patch</th>
      </tr>
      <tr>
        <td style='border:1px solid #888;padding:6px;vertical-align:top;'><pre style='white-space:pre-wrap;'>{esc(flat_code)}</pre></td>
        <td style='border:1px solid #888;padding:6px;vertical-align:top;'><pre style='white-space:pre-wrap;'>{esc(belief_code)}</pre></td>
      </tr>
    </table>
    """
    display(HTML(html))

show_patches_side_by_side(
    flat_transcript[-1]["code"],
    belief_result["patch"],
)


In [ ]:
# Summary table: hard tests passing, soft-preference average score, # attempts.
flat_tests = flat_transcript[-1]["test"]
belief_tests = belief_result["test"]

# Score flat agent's patch against the same preferences for an apples-to-apples
# preference comparison.
flat_pref_scores = judge_patch(flat_transcript[-1]["code"], PAIR_A, TASK)

def mean_pref(scores: dict) -> float:
    return float(np.mean([s["score"] for s in scores.values()]))

summary = {
    "flat agent":  {
        "tests": f"{flat_tests['n_passed']}/{flat_tests['n_total']}",
        "all_passed": flat_tests["all_passed"],
        "n_attempts": len(flat_transcript),
        "mean_pref_score": mean_pref(flat_pref_scores),
        "clarifications": 0,
    },
    "belief-driven agent": {
        "tests": f"{belief_tests['n_passed']}/{belief_tests['n_total']}",
        "all_passed": belief_tests["all_passed"],
        "n_attempts": 1,  # belief agent does not retry within this loop
        "mean_pref_score": mean_pref(belief_result["pref_scores"]),
        "clarifications": len(belief_result["clarifications"]),
    },
}

display(Markdown(
    "| | tests | all_passed | attempts | clarifications | mean preference score |\n"
    "|---|---|---|---|---|---|\n" +
    "\n".join(
        f"| **{k}** | {v['tests']} | {'check' if v['all_passed'] else 'cross'} "
        f"| {v['n_attempts']} | {v['clarifications']} | {v['mean_pref_score']:.2f} |"
        .replace("check", "✓").replace("cross", "✗")
        for k, v in summary.items())
))


## 9. Emergent conflict detection on the full preference list

So far the scripted run used only the two preferences engineered to conflict. The interesting question is whether the framework finds the same conflict when those two are buried among a larger list of distractors, preferences that are either irrelevant to this task or non-conflicting.

We re-run the belief agent on all eight preferences and inspect what the conflict detector picks out.


In [ ]:
print("Running belief-driven agent on PREFS_ALL (8 preferences)...")
print()
belief_result_full = belief_agent(TASK, PREFS_ALL, ALEX_HISTORY,
                                    max_clarifications=2, verbose=True)
print()
print(f"Tests: {belief_result_full['test']['n_passed']}/"
      f"{belief_result_full['test']['n_total']}.")
print(f"Clarifications used: {len(belief_result_full['clarifications'])}.")
for cl in belief_result_full["clarifications"]:
    print(f"  Round {cl['round']}: {cl['pair']}")


In [ ]:
# Compatibility matrix: ask the judge for pairwise compatibility scores
# across all pairs in PREFS_ALL, then render as a heatmap. This visualises
# *why* the conflict detector flagged the pair it did.

PAIR_JUDGE_SYSTEM = (
    "You score pairwise compatibility of soft coding preferences on a given "
    "task: 1 = both can be fully satisfied simultaneously, 0 = they "
    "fundamentally conflict on this task. Output JSON: "
    '{"compatibility": <0..1>, "rationale": "<one sentence>"}.'
)

def pair_compatibility(a: Constraint, b: Constraint, task: CodingTask) -> float:
    prompt = (
        f"TASK: {task.problem_statement}\n\n"
        f"PREFERENCE A: {a.statement}\n"
        f"PREFERENCE B: {b.statement}\n\n"
        "Score compatibility on this specific task."
    )
    res = llm_json(prompt, system=PAIR_JUDGE_SYSTEM, max_tokens=200)
    return float(res.get("compatibility", 0.5))


def plot_pair_compatibility(constraints: list[Constraint], task: CodingTask):
    n = len(constraints)
    M = np.ones((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            score = pair_compatibility(constraints[i], constraints[j], task)
            M[i, j] = score
            M[j, i] = score
    names = [c.name for c in constraints]
    fig, ax = plt.subplots(figsize=(0.7 * n + 2, 0.7 * n + 1))
    im = ax.imshow(M, cmap="RdYlGn", vmin=0, vmax=1)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(names, fontsize=8)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center",
                    fontsize=7,
                    color="black" if M[i,j] > 0.5 else "white")
    fig.colorbar(im, ax=ax, fraction=0.03, label="compatibility")
    ax.set_title("Pairwise preference compatibility on this task", fontsize=10)
    plt.tight_layout()
    plt.show()
    return M

# This call is the most expensive cell in the notebook (n*(n-1)/2 LLM calls).
# Comment out if you want to skip during iteration.
_M = plot_pair_compatibility(PREFS_ALL, TASK)


## 10. Visualising the fix

We apply the belief-driven agent's patch to the live `Colorbar` class and re-render the colorbar. The dividers at the extended ends now appear, the very bug from PR #22865, fixed.


In [ ]:
# The canonical gold patch (GOLD_ADD_SOLIDS_SRC) was defined in the imports
# cell. We reuse it here as a robust fallback when the agent's patch
# references API not present in the installed matplotlib.

def safe_patch_for_render(agent_patch: str) -> str:
    """Use the agent's patch if it executes cleanly on a render; otherwise
    fall back to the canonical gold patch. Always restores known-good state
    before returning so the caller can sequence install_bug / apply_patch
    deterministically."""
    try:
        apply_patch(agent_patch)
        fig_probe, ax_probe = plt.subplots()
        _ = mcb.ColorbarBase(
            ax_probe, cmap=plt.get_cmap("viridis", 5),
            norm=matplotlib.colors.BoundaryNorm(np.arange(6), 5),
            boundaries=[-1, 0, 1, 2, 3, 4, 5, 6],
            extend="both", drawedges=True, spacing="uniform",
        )
        plt.close(fig_probe)
        return agent_patch
    except Exception as exc:
        print(f"[note] agent patch did not render cleanly ({type(exc).__name__}: "
              f"{str(exc)[:80]}); falling back to gold patch for the visual.")
        return GOLD_ADD_SOLIDS_SRC
    finally:
        restore_fix()

# Snapshot state, install the bug, render the broken colorbar, install the
# patch we'll actually use, render the fixed colorbar side-by-side.
render_patch = safe_patch_for_render(belief_result["patch"])

restore_fix()
install_bug()

fig, axes = plt.subplots(1, 2, figsize=(4, 4),
                          gridspec_kw={"wspace": 0.5})
for ax, title in zip(axes, ["before (buggy)", "after (agent patch)"]):
    boundaries = np.array([0, 1, 2, 3, 4, 5])
    cmap = plt.get_cmap("viridis", len(boundaries) - 1)
    norm = matplotlib.colors.BoundaryNorm(boundaries, cmap.N)
    if title.startswith("after"):
        apply_patch(render_patch)
    cb = mcb.ColorbarBase(
        ax, cmap=cmap, norm=norm,
        boundaries=[-1] + list(boundaries) + [6],
        extend="both", drawedges=True, spacing="uniform",
    )
    ax.set_title(title, fontsize=9)
plt.show()

# Restore the real fixed implementation for any later code.
restore_fix()


## 11. Analysis

### What changed between the flat and belief-driven agents

| Failure mode in flat agent | Mechanism the belief agent adds |
| --- | --- |
| Picks a patch shape that violates the user's tacit style preference | `read_session_history` initialises a non-uniform belief informed by prior turns; the patch generator sees preferences in posterior order |
| Cannot detect that two preferences conflict; silently picks one | `detect_conflict` (an LLM judge) flags pairs that would produce visibly different patches and asks one focused question |
| If it loops on test failure, may regress by overfitting to the last error | The belief agent does not iterate the patch in this loop: by the time `execute_code_action` fires, all preconditions are met and one attempt is the design point. (Iteration would be added at the OpenTAMP symbolic-replanning layer, not at the continuous layer.) |
| No mechanism to surface *why* a patch satisfies or violates a preference | `judge_patch` produces per-preference scores with rationales, the soft-constraint analogue of the hard test signal |

### Mapping back to OpenTAMP

| This notebook | OpenTAMP / SWE-TAMP equivalent |
| --- | --- |
| `Constraint`, `CodingTask`, `AgentState`, `ParticleBelief` | Types and predicates in `swe_tamp_meta.json` |
| `read_session_history`, `clarify_constraint`, `clarify_intent`, `confirm_spec`, `execute_code_action` | Actions in `swe_tamp_acts.json` |
| `detect_conflict` | The belief-logic override (`swe_belief_domain_logic.py`), analogous to `belief_domain_logic_pop.py` for nav_2d |
| The action skeleton plot | The output of FastDownward symbolic planning, before continuous refinement |
| `run_test_against` | The continuous-layer feasibility check |
| Pytest exit code | The hard-constraint goal predicate `TestsPassing(t)` |
| `judge_patch` per-preference scores | The soft-cost term over preference predicates |

### What is missing for the full SWE-TAMP integration

1. **Symbolic planner wiring.** FastDownward needs the action schemas in PDDL form; the JSON specs in `opentamp/new_specs/swe_domain_belief/` are not yet authored.
2. **Backtracking on failure.** This notebook treats `execute_code_action` as a one-shot. The full integration routes test failures back to the symbolic layer, which can re-plan (for example, by re-issuing `clarify_constraint` for a different pair, or by changing the proposed milestone).
3. **Real prior-session storage.** ToM-SWE's 453 raw developer sessions are not public; we used a hand-authored profile. The full pipeline would index real sessions and retrieve relevant turns per task.
4. **Conflict-resolution learning.** Right now the user oracle answers the question; in a real deployment we would learn a policy that maps task features to resolution preferences, so the agent asks less over time.


## 12. Bonus: framework generalisation to `astropy__astropy-12907`

The astropy task involves `separability_matrix` returning incorrect separability for nested `CompoundModels`. The fix admits two visibly different implementations (recursion over the model tree vs. iteration with a stack), creating a preference conflict between *prefer simple control flow* and *match existing recursive pattern in the module*.

The same `belief_agent` function applies unchanged. To run this case end-to-end:

```python
ASTROPY_TASK = CodingTask(
    instance_id="astropy__astropy-12907",
    problem_statement=(
        "astropy.modeling.separable.separability_matrix returns an incorrect "
        "matrix for nested CompoundModels. The function should recursively "
        "compute separability for each sub-model and combine the results "
        "according to the compound model operator."
    ),
    target_method="astropy.modeling.separable._cstack",
    failing_test="...",  # the SWE-bench Verified parametrized test
)

belief_result_astropy = belief_agent(
    ASTROPY_TASK, PREFS_ALL, ALEX_HISTORY, max_clarifications=2,
)
```

We leave the full astropy run as future work, its harness requires a heavier mock since the bug spans two helper functions rather than living in one method body. The framework itself is unchanged.


## 13. References and next steps

**Primary references**

- Wang et al., *ToM-SWE: User Mental Modeling for Software Engineering Agents*, arXiv:2510.21903 (2025).
- Jimenez et al., *SWE-bench: Can Language Models Resolve Real-World GitHub Issues?*, ICLR 2024. See also the [SWE-bench Verified](https://www.swebench.com) subset.
- The OpenTAMP framework: [Algorithmic-Alignment-Lab/openTAMP](https://github.com/Algorithmic-Alignment-Lab/openTAMP).
- The specific task instance: [matplotlib PR #22865](https://github.com/matplotlib/matplotlib/pull/22865) and [issue #22864](https://github.com/matplotlib/matplotlib/issues/22864).

**Files to author next, in the openTAMP repo**

```
opentamp/new_specs/swe_domain_belief/
  swe_tamp_meta.json
  swe_tamp_acts.json
  swe_tamp_prob.json
  swe_env_hyperparam.py

opentamp/minimal_tamp/
  swe_belief_domain_logic.py
```

These mirror the layout of `opentamp/new_specs/nav_domain_belief/` and `opentamp/minimal_tamp/belief_domain_logic_pop.py`. The dataclasses and action functions in this notebook map mechanically into those files; the planning loop in [`backtrack_plan_recursive.py`](minimal_tamp/backtrack_plan_recursive.py) consumes them without modification.
